# gemini — a usage shape that shares nothing with OpenAI

There is no `usage`; there is `usage_metadata`. And the streaming method reports usage **cumulatively** — each chunk carries the running total.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## The five steps

Every recipe in `providers/` walks the same five, in the same order:

| # | Step | Here |
|---|---|---|
| 1 | **connect** | `genai.Client()` — the `models.generate_content` shape |
| 2 | **instrument** | one wrap — detection is structural, not name-based |
| 3 | **govern** | a `tokenguard` budget **and** a `guardrails` gate |
| 4 | **record** | `cassette` — the same call replayed offline, 0 provider calls |
| 5 | **prove** | `acttrace` `verify()` and a cost that came from `prices` |

**Distinctive here: `usage_metadata`, and a cumulative stream.** Summing `candidates_token_count` across chunks triple-counts a three-chunk answer.

## 1–3 · Connect, instrument, call

In [ ]:
import main as recipe
from cendor.core import bus, instrument
from cendor.core.types import LLMCall

seen = []
bus.subscribe(lambda e: seen.append(e) if isinstance(e, LLMCall) else None)
client = instrument(recipe.fake_genai())
recipe.ask(client, "Is this request within policy?")
call = seen[-1]
print(f"provider: {call.provider}   (inferred from the client's shape)")
print(f"usage   : {call.usage.input_tokens} in + {call.usage.output_tokens} out")
print(f"cost    : ${call.cost.amount}")

## The stream, and the trap in it

Three chunks reporting 70 → 140 → 210 are **one** 210-token answer, not a 420-token one. `instrument()` takes the last.

In [ ]:
before = len(seen)
chunks = list(client.models.generate_content_stream(model=recipe.MODEL, contents="Stream it."))
stream_call = seen[-1]
print(f"{len(chunks)} chunks -> {stream_call.usage.output_tokens} out")

## 5 · Prove it

In [ ]:
assert call.usage.input_tokens == 980, "usage_metadata was not normalized"
assert call.cost and call.cost.amount > 0
assert stream_call.usage.output_tokens == 210, "cumulative stream usage was summed"
print("OK")